In [1]:
from google.colab import drive
import shutil


drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Data_Images_processed.zip'
extract_to = '/content/Data_Images_processed'


shutil.unpack_archive(zip_path, extract_to, 'zip')


Mounted at /content/drive


# SentinelFrame — LoRA Fine-tuning of OpenCLIP ViT-B/32

Continuation of `preprocess_pipeline.ipynb`: trains a multi-label classifier on the 224x224 crops in `Data_Images_processed/{label}/`, using a frozen OpenCLIP vision tower with LoRA adapters plus a new classification head.

**A few corrections made relative to the pasted reference snippet, worth knowing about:**
- `NUM_LABELS`/`LABELS` are imported directly from `generate_images.py` instead of re-typed, so this notebook can't silently drift out of sync if the label set changes (the reference snippet's hardcoded 7 labels were already stale — the project has 8 now).
- LoRA `target_modules` are discovered from the actual model at runtime instead of assumed. OpenCLIP's `ResidualAttentionBlock` uses `nn.MultiheadAttention` (a single fused QKV projection, no separate `q_proj`/`k_proj`/`v_proj` submodules) plus an MLP with `c_fc`/`c_proj` — the reference's `["q_proj", "k_proj", "v_proj", "out_proj"]` would only ever match `out_proj`, silently applying LoRA to a fraction of what was intended.
- The head has no final `Sigmoid` — `BCEWithLogitsLoss` expects raw logits and applies a numerically-stable sigmoid internally; an explicit `Sigmoid` layer would double up. Probabilities are computed with `torch.sigmoid()` only where needed (metrics/inference).
- `DEVICE` auto-detects CUDA instead of hardcoding `"cuda"`, so this doesn't crash on a CPU-only machine.
- The optimizer only receives `requires_grad=True` parameters (LoRA + head), the standard practice for a frozen-backbone setup.

In [2]:
!pip install -q open_clip_torch peft scikit-learn pillow tqdm
# If torch/torchvision aren't already installed (e.g. a fresh environment), uncomment —
# leave commented if you already have a CUDA-matched torch build, to avoid clobbering it:
# !pip install -q torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00


In [ ]:
import sys
import gc
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import open_clip
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, accuracy_score
from tqdm.auto import tqdm

sys.path.insert(0, "/content")
LABELS = [
    "hand_in_pocket",
    "hand_in_bag",
    "hand_under_clothing",
    "object_in_hand",
    "interacting_with_shelf",
    "hand_occluded_generic",
    "both_hands_not_visible",
    "no_visible_hand_interaction",
]

In [4]:
# ============================================================
# Configuration
# ============================================================

DATA_DIR = Path("/content/Data_Images_processed")
CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

NUM_LABELS = len(LABELS)
IMAGE_SIZE = 224
EMBED_DIM = 512  # ViT-B/32's encode_image() output dimension

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_SEED = 42

BATCH_SIZE = 16
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4

# The full dataset (~160 images) is preloaded into RAM once (see the Dataset cell
# below), so DataLoader worker subprocesses buy nothing here and were the actual
# cause of the "DataLoader worker killed" OOM on Colab: with num_workers>0 and
# persistent_workers defaulting to False, PyTorch respawns worker processes every
# epoch, and repeated forking of a growing parent process steadily inflates host
# RAM. 0 workers means the main process just indexes an in-memory tensor directly.
NUM_WORKERS = 0

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("⚠ No CUDA GPU detected — fine-tuning will be slow on CPU.")
print(f"Labels ({NUM_LABELS}): {LABELS}")

Device: cuda
Labels (8): ['hand_in_pocket', 'hand_in_bag', 'hand_under_clothing', 'object_in_hand', 'interacting_with_shelf', 'hand_occluded_generic', 'both_hands_not_visible', 'no_visible_hand_interaction']


## 1. Discover processed images

In [5]:
image_paths, image_label_names = [], []

for label_dir in sorted(DATA_DIR.iterdir()):
    if not label_dir.is_dir():
        continue
    if label_dir.name not in LABELS:
        print(f"⚠ Skipping unknown directory: {label_dir.name}")
        continue
    for img_path in sorted(label_dir.glob("*.png")):
        image_paths.append(img_path)
        image_label_names.append(label_dir.name)

print(f"Found {len(image_paths)} processed images across {len(set(image_label_names))} label(s):")
for label in LABELS:
    print(f"  - {label}: {image_label_names.count(label)}")

Found 1920 processed images across 8 label(s):
  - hand_in_pocket: 240
  - hand_in_bag: 240
  - hand_under_clothing: 240
  - object_in_hand: 240
  - interacting_with_shelf: 240
  - hand_occluded_generic: 240
  - both_hands_not_visible: 240
  - no_visible_hand_interaction: 240


## 2. Train / val / test split

Each processed image is single-label (one behavior per generated image), so the split is stratified directly on that label to keep class balance across the three sets.

In [6]:
import re

# ------------------------------------------------------------------
# Leakage-safe split.
#
# The preprocessing notebook wrote augmented copies of every source image
# (named "<stem>_aug_<NNN>.png") into the SAME per-label folders as the
# originals. Splitting the flat image list directly would scatter an original
# and its augmentations across train/val/test, leaking near-duplicates and
# inflating val/test scores.
#
# Fix, in two parts:
#   1. Split at the level of ORIGINAL images, so an original and all its
#      augmentations stay together in one split.
#   2. Keep augmentations only in TRAIN. Val and test are evaluated on the clean,
#      un-augmented originals — the distribution we actually care about.
#
# Two subtleties this handles:
#   * The augmentation suffix is matched loosely ("_aug" followed by optional
#     "_" and digits), so both "sample_1_aug1" and "sample_1_aug_001" work.
#   * Sample numbers REPEAT across label folders (each label has its own
#     sample_001..sample_020), so the grouping key is (label, original_stem),
#     NOT the bare stem — otherwise every label's "sample_001" collides and the
#     whole dataset collapses into a single split.
# ------------------------------------------------------------------

AUG_RE = re.compile(r"_aug_?\d+$")


def is_augmented(path):
    return bool(AUG_RE.search(path.stem))


def group_key(path):
    """(label, original-stem) — identifies the source image an original or any of
    its augmentations came from. Includes the label so that identical sample
    numbers in different label folders don't collide."""
    return (path.parent.name, AUG_RE.sub("", path.stem))


label_indices = [LABELS.index(name) for name in image_label_names]

# One entry per ORIGINAL (non-augmented) source image, to drive a stratified
# split over originals only.
orig_paths, orig_label_idx = [], []
for path, lbl in zip(image_paths, label_indices):
    if not is_augmented(path):
        orig_paths.append(path)
        orig_label_idx.append(lbl)

assert orig_paths, "No non-augmented originals found — check the AUG_RE suffix pattern."

train_val_orig, test_orig, train_val_oidx, test_oidx = train_test_split(
    orig_paths, orig_label_idx,
    test_size=TEST_FRAC, stratify=orig_label_idx, random_state=RANDOM_SEED,
)
val_relative_frac = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
train_orig, val_orig, train_oidx, val_oidx = train_test_split(
    train_val_orig, train_val_oidx,
    test_size=val_relative_frac, stratify=train_val_oidx, random_state=RANDOM_SEED,
)

# Which source images (by (label, stem) key) belong to each split.
train_keys = {group_key(p) for p in train_orig}
val_keys = {group_key(p) for p in val_orig}
test_keys = {group_key(p) for p in test_orig}

# Assign every actual file (originals + augmentations) to its split.
#   - train  : the original + all its augmentations
#   - val/test: the original only (augmentations dropped)
train_paths, train_idx = [], []
val_paths, val_idx = [], []
test_paths, test_idx = [], []

for path, lbl in zip(image_paths, label_indices):
    key = group_key(path)
    if key in train_keys:
        train_paths.append(path)
        train_idx.append(lbl)
    elif key in val_keys:
        if not is_augmented(path):            # keep val clean
            val_paths.append(path)
            val_idx.append(lbl)
    elif key in test_keys:
        if not is_augmented(path):            # keep test clean
            test_paths.append(path)
            test_idx.append(lbl)

n_train_aug = sum(is_augmented(p) for p in train_paths)
print(f"Originals: {len(orig_paths)}   "
      f"Train: {len(train_paths)} ({n_train_aug} augmented)   "
      f"Val: {len(val_paths)}   Test: {len(test_paths)}")
print("Val/Test contain no augmented images; no original is split across sets.")

with open(CHECKPOINT_DIR / "splits.json", "w") as f:
    json.dump({
        "train": [str(p) for p in train_paths],
        "val": [str(p) for p in val_paths],
        "test": [str(p) for p in test_paths],
    }, f, indent=2)


def to_multihot(idx):
    vec = np.zeros(NUM_LABELS, dtype=np.float32)
    vec[idx] = 1.0
    return vec


train_labels = [to_multihot(i) for i in train_idx]
val_labels = [to_multihot(i) for i in val_idx]
test_labels = [to_multihot(i) for i in test_idx]

Originals: 320   Train: 1344 (1120 augmented)   Val: 48   Test: 48
Val/Test contain no augmented images; no original is split across sets.


## 3. Load OpenCLIP and extract just its Normalize transform

Loaded once here and reused later for the classifier — no need to load the weights twice. Only `Normalize` is pulled out of `preprocess`; `Resize`/`CenterCrop` are skipped since the preprocessing notebook already produced exact 224x224 crops.

In [7]:
clip_model, clip_preprocess, _ = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)

normalize = next(t for t in clip_preprocess.transforms if isinstance(t, transforms.Normalize))
print(f"Extracted Normalize: mean={normalize.mean}, std={normalize.std}")

image_transform = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Extracted Normalize: mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)


## 4. PyTorch Dataset + DataLoaders

The whole dataset is small enough (~160 images at 224x224) to fit comfortably in RAM, so every image is decoded and transformed once up front in `__init__` rather than re-reading and re-transforming from disk on every `__getitem__` call across every epoch. `__getitem__` then just indexes an already-materialized tensor — cheap enough that `num_workers=0` (no subprocess workers) is the right call, sidestepping the DataLoader worker OOM entirely.

In [8]:
class CCTVDataset(Dataset):
    def __init__(self, image_paths, label_vectors, transform):
        images = []
        for path in image_paths:
            img = Image.open(path).convert("RGB")
            assert img.size == (IMAGE_SIZE, IMAGE_SIZE), (
                f"{path} is {img.size}, expected ({IMAGE_SIZE}, {IMAGE_SIZE}) — "
                f"run the preprocessing notebook first."
            )
            images.append(transform(img))

        self.images = torch.stack(images)  # [N, 3, IMAGE_SIZE, IMAGE_SIZE], preloaded once
        self.labels = torch.from_numpy(np.stack(label_vectors))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [9]:
train_dataset = CCTVDataset(train_paths, train_labels, image_transform)
val_dataset = CCTVDataset(val_paths, val_labels, image_transform)
test_dataset = CCTVDataset(test_paths, test_labels, image_transform)
print(f"Preloaded {len(train_dataset)} train / {len(val_dataset)} val / {len(test_dataset)} test images into memory")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

print(f"Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")

Preloaded 1344 train / 48 val / 48 test images into memory
Batches — train: 84  val: 3  test: 3


## 5. Freeze the backbone and inject LoRA

In [10]:
!pip install --upgrade torchao

clip_model = clip_model.to(DEVICE)

for p in clip_model.parameters():
    p.requires_grad = False


def discover_lora_targets(module, suffixes=("out_proj", "c_fc", "c_proj")):
    found = set()
    for name, sub in module.named_modules():
        if isinstance(sub, nn.Linear) and name.split(".")[-1] in suffixes:
            found.add(name.split(".")[-1])
    return sorted(found)


lora_target_modules = discover_lora_targets(clip_model.visual)
print(f"LoRA target modules: {lora_target_modules}")
assert lora_target_modules, "No Linear submodules matched — inspect clip_model.visual's structure."

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=lora_target_modules,
    lora_dropout=LORA_DROPOUT,
    bias="none",
)

clip_model.visual = get_peft_model(clip_model.visual, lora_config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


LoRA target modules: ['c_fc', 'c_proj', 'out_proj']


## 6. Multi-label classification head

In [11]:
class Classifier(nn.Module):
    def __init__(self, base_model, embed_dim, num_labels):
        super().__init__()
        self.base = base_model
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def forward(self, x):
        features = self.base.encode_image(x).float()
        return self.head(features)


model = Classifier(clip_model, EMBED_DIM, NUM_LABELS).to(DEVICE)

with torch.no_grad():
    dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    dummy_out = model(dummy)
assert dummy_out.shape == (1, NUM_LABELS), f"Unexpected output shape {dummy_out.shape}"

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Trainable params: 1,018,120 / 152,295,433 (0.67%)


## 7. Loss, optimizer, metrics

Each label is positive in only ~1/`NUM_LABELS` of examples (every generated image is single-label), so unweighted `BCEWithLogitsLoss` lets the model minimize loss by pushing every logit toward "negative" — it gets the ~87.5% of true negatives right for free and the ~12.5% true positives barely move the average. `pos_weight` (computed from the actual train-set class balance, not guessed) counteracts this by weighting each class's positive term by its negative:positive ratio.

In [12]:
train_labels_arr = np.stack(train_labels)
num_pos = train_labels_arr.sum(axis=0)
num_neg = len(train_labels_arr) - num_pos
pos_weight = torch.tensor(num_neg / np.clip(num_pos, 1, None), dtype=torch.float32, device=DEVICE)

print("Per-label pos_weight (train set):")
for label, w, p in zip(LABELS, pos_weight.tolist(), num_pos):
    print(f"  - {label}: pos_weight={w:.2f}  ({int(p)} positive / {len(train_labels_arr)} total)")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)


def compute_metrics(y_true, y_logits, threshold=0.5):
    y_prob = torch.sigmoid(torch.as_tensor(y_logits)).numpy()
    y_pred = (y_prob >= threshold).astype(np.float32)
    y_true = np.asarray(y_true)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    try:
        map_score = average_precision_score(y_true, y_prob, average="macro")
    except ValueError:
        map_score = float("nan")  # a class with zero positives in this batch/split

    return {
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "mAP_macro": float(map_score),
        "subset_accuracy": float(accuracy_score(y_true, y_pred)),
    }

Per-label pos_weight (train set):
  - hand_in_pocket: pos_weight=7.00  (168 positive / 1344 total)
  - hand_in_bag: pos_weight=7.00  (168 positive / 1344 total)
  - hand_under_clothing: pos_weight=7.00  (168 positive / 1344 total)
  - object_in_hand: pos_weight=7.00  (168 positive / 1344 total)
  - interacting_with_shelf: pos_weight=7.00  (168 positive / 1344 total)
  - hand_occluded_generic: pos_weight=7.00  (168 positive / 1344 total)
  - both_hands_not_visible: pos_weight=7.00  (168 positive / 1344 total)
  - no_visible_hand_interaction: pos_weight=7.00  (168 positive / 1344 total)


## 8. Train / validate, with checkpointing

The best model (by validation macro-F1) and a running `training_history.json` are saved after every epoch, so a long run isn't lost if interrupted.

In [13]:
def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    all_labels, all_logits = [], []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_logits.append(logits.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(np.concatenate(all_labels), np.concatenate(all_logits))
    metrics["loss"] = avg_loss

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return metrics

In [14]:
history = []
best_val_f1 = -1.0
best_ckpt_path = CHECKPOINT_DIR / "best_model.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)

    print(f"[Epoch {epoch}/{EPOCHS}] "
          f"train_loss={train_metrics['loss']:.4f} val_loss={val_metrics['loss']:.4f} "
          f"val_f1={val_metrics['f1_macro']:.4f} val_mAP={val_metrics['mAP_macro']:.4f}")

    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    with open(CHECKPOINT_DIR / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    if val_metrics["f1_macro"] > best_val_f1:
        best_val_f1 = val_metrics["f1_macro"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_f1_macro": best_val_f1,
        }, best_ckpt_path)
        print(f"  ✓ New best model saved (val_f1_macro={best_val_f1:.4f}) -> {best_ckpt_path}")

print(f"\nBest val_f1_macro: {best_val_f1:.4f}  (checkpoint: {best_ckpt_path})")

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[Epoch 1/100] train_loss=1.1390 val_loss=1.0041 val_f1=0.4710 val_mAP=0.6749
  ✓ New best model saved (val_f1_macro=0.4710) -> /content/checkpoints/best_model.pt


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

...

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[Epoch 37/100] train_loss=0.0003 val_loss=0.8913 val_f1=0.8477 val_mAP=0.9128


  0%|          | 0/84 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 9. Final test-set evaluation

In [15]:
checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} "
      f"(val_f1_macro={checkpoint['val_f1_macro']:.4f})")

test_metrics = run_epoch(test_loader, train=False)
print("\nTest set results:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

with open(CHECKPOINT_DIR / "test_results.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

Loaded best checkpoint from epoch 16 (val_f1_macro=0.8783)


  0%|          | 0/3 [00:00<?, ?it/s]


Test set results:
  precision_macro: 0.8214
  recall_macro: 0.7917
  f1_macro: 0.8002
  mAP_macro: 0.8568
  subset_accuracy: 0.7500
  loss: 0.9869


# 10. Model comparison — DINOv2, MobileNetV3, EfficientNetV2, SigLIP, ConvNeXt V2, ResNet-18

Each cell below is **self-contained**: it re-uses the leakage-safe split
(`train_paths` / `val_paths` / `test_paths` and the matching `*_labels`),
`compute_metrics`, `CCTVDataset`, and the config constants (`DEVICE`, `EPOCHS`,
`BATCH_SIZE`, `LR`, `WEIGHT_DECAY`, `CHECKPOINT_DIR`, `LORA_*`) from the cells
above, then loads its own backbone with its own preprocessing, trains, and
evaluates on the test set — mirroring the OpenCLIP pipeline so the numbers are
directly comparable (same data, head, loss, optimizer, epochs).

Run the CLIP notebook above (through at least the split + metrics cells) once
before running these. Each writes its own `best_model_<name>.pt` /
`test_results_<name>.json` / `training_history_<name>.json`, so they don't clash
with the CLIP artifacts or each other.

**Pretrained + frozen backbone, LoRA-adapted (ViT-style, the literal CLIP repeat):**
- **DINOv2 ViT-B/14** — LoRA on `qkv`/`proj`/`fc1`/`fc2`.
- **SigLIP ViT-B/16** — LoRA on the vision tower's attention + MLP linears.

**Pretrained + frozen backbone + head (the standard CNN transfer-learning setup):**
- **MobileNetV3-Large**, **EfficientNetV2-S** (torchvision), **ConvNeXt V2-Tiny** (timm).

**From scratch (baseline — no pretrained weights, fully trainable):**
- **ResNet-18** — same data/head/loss/optimizer/epochs, random init. The gap
  between this and the pretrained models is the "value of pretraining" result;
  expect it to score lower on so few images.

In [16]:
# ================================================================
# 10a. DINOv2 ViT-B/14 — frozen backbone + LoRA + head
# ================================================================
# Self-contained: reuses train/val/test_paths, *_labels, compute_metrics,
# CCTVDataset, and the config constants from the CLIP cells above.

# DINOv2 is distributed via torch.hub; xformers is optional (falls back to
# vanilla attention if absent).
!pip install -q xformers

import copy
import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

MODEL_NAME = "dinov2_vitb14"
DINOV2_EMBED_DIM = 768           # ViT-B CLS-token dimension
# DINOv2 patch size is 14, so the input side must be a multiple of 14.
# 224 = 14 * 16, so our existing 224x224 crops work unchanged.
assert IMAGE_SIZE % 14 == 0, "DINOv2 needs an input size divisible by 14."

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Model-specific preprocessing (ImageNet stats, NOT CLIP's) -------------
dinov2_normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
)
dinov2_transform = transforms.Compose([transforms.ToTensor(), dinov2_normalize])

# The processed crops are already 224x224, so no Resize/CenterCrop needed —
# same assumption the CLIP path makes.
dino_train_ds = CCTVDataset(train_paths, train_labels, dinov2_transform)
dino_val_ds   = CCTVDataset(val_paths,   val_labels,   dinov2_transform)
dino_test_ds  = CCTVDataset(test_paths,  test_labels,  dinov2_transform)

dino_train_loader = DataLoader(dino_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
dino_val_loader   = DataLoader(dino_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
dino_test_loader  = DataLoader(dino_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

# ---- Load pretrained backbone, freeze it, inject LoRA ----------------------
dino_backbone = torch.hub.load("facebookresearch/dinov2", MODEL_NAME)
for p in dino_backbone.parameters():
    p.requires_grad = False

# DINOv2 blocks use fused `qkv` + `proj` attention and `fc1`/`fc2` MLP linears —
# the ViT analogue of CLIP's out_proj/c_fc/c_proj. Discover from the model so we
# don't hardcode names that could drift.
def _discover_dino_lora_targets(module, suffixes=("qkv", "proj", "fc1", "fc2")):
    found = set()
    for name, sub in module.named_modules():
        if isinstance(sub, nn.Linear) and name.split(".")[-1] in suffixes:
            found.add(name.split(".")[-1])
    return sorted(found)

dino_targets = _discover_dino_lora_targets(dino_backbone)
print(f"DINOv2 LoRA target modules: {dino_targets}")
assert dino_targets, "No Linear submodules matched — inspect dino_backbone structure."

dino_lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=dino_targets,
    lora_dropout=LORA_DROPOUT, bias="none",
)
dino_backbone = get_peft_model(dino_backbone, dino_lora_cfg)


class DINOv2Classifier(nn.Module):
    """Frozen (LoRA-adapted) DINOv2 backbone + multi-label head.
    Backbone forward returns the CLS embedding [B, 768]."""
    def __init__(self, backbone, embed_dim, num_labels):
        super().__init__()
        self.base = backbone
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def forward(self, x):
        features = self.base(x).float()      # dinov2 __call__ -> CLS token [B, 768]
        return self.head(features)


dino_model = DINOv2Classifier(dino_backbone, DINOV2_EMBED_DIM, NUM_LABELS).to(DEVICE)

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    assert dino_model(_dummy).shape == (1, NUM_LABELS)

_trainable = sum(p.numel() for p in dino_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in dino_model.parameters())
print(f"[DINOv2] Trainable params: {_trainable:,} / {_total:,} "
      f"({100 * _trainable / _total:.2f}%)")

# ---- Loss (pos_weight from THIS split's train balance), optimizer ----------
_train_arr = np.stack(train_labels)
_num_pos = _train_arr.sum(axis=0)
_num_neg = len(_train_arr) - _num_pos
dino_pos_weight = torch.tensor(_num_neg / np.clip(_num_pos, 1, None),
                               dtype=torch.float32, device=DEVICE)
dino_criterion = nn.BCEWithLogitsLoss(pos_weight=dino_pos_weight)
dino_optimizer = torch.optim.AdamW(
    (p for p in dino_model.parameters() if p.requires_grad),
    lr=LR, weight_decay=WEIGHT_DECAY,
)


# ---- Train / validate with checkpointing (mirrors the CLIP loop) -----------
def dino_run_epoch(loader, train: bool):
    dino_model.train(train)
    total_loss = 0.0
    all_labels, all_logits = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = dino_model(imgs)
            loss = dino_criterion(logits, labels)
            if train:
                dino_optimizer.zero_grad()
                loss.backward()
                dino_optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_logits.append(logits.detach().cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_labels), np.concatenate(all_logits))
    metrics["loss"] = total_loss / len(loader.dataset)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics


dino_history = []
dino_best_f1 = -1.0
dino_best_ckpt = CHECKPOINT_DIR / "best_model_dinov2.pt"

for epoch in range(1, EPOCHS + 1):
    tr = dino_run_epoch(dino_train_loader, train=True)
    va = dino_run_epoch(dino_val_loader, train=False)
    print(f"[DINOv2][Epoch {epoch}/{EPOCHS}] "
          f"train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} "
          f"val_f1={va['f1_macro']:.4f} val_mAP={va['mAP_macro']:.4f}")
    dino_history.append({"epoch": epoch, "train": tr, "val": va})
    with open(CHECKPOINT_DIR / "training_history_dinov2.json", "w") as f:
        json.dump(dino_history, f, indent=2)
    if va["f1_macro"] > dino_best_f1:
        dino_best_f1 = va["f1_macro"]
        torch.save({"epoch": epoch, "model_state_dict": dino_model.state_dict(),
                    "val_f1_macro": dino_best_f1}, dino_best_ckpt)
        print(f"  ✓ New best (val_f1_macro={dino_best_f1:.4f}) -> {dino_best_ckpt}")

print(f"\n[DINOv2] Best val_f1_macro: {dino_best_f1:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.6 MB/s eta 0:00:00
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:03<00:00, 100MB/s]


DINOv2 LoRA target modules: ['fc1', 'fc2', 'proj', 'qkv']
[DINOv2] Trainable params: 1,389,416 / 87,969,896 (1.58%)


...

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[DINOv2][Epoch 30/100] train_loss=0.0001 val_loss=0.2070 val_f1=0.9451 val_mAP=0.9842


  0%|          | 0/84 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [17]:
# ---- Final test-set evaluation ---------------------------------------------
_ckpt = torch.load(dino_best_ckpt, map_location=DEVICE)
dino_model.load_state_dict(_ckpt["model_state_dict"])
print(f"[DINOv2] Loaded best checkpoint from epoch {_ckpt['epoch']} "
      f"(val_f1_macro={_ckpt['val_f1_macro']:.4f})")

dino_test_metrics = dino_run_epoch(dino_test_loader, train=False)
print("\n[DINOv2] Test set results:")
for k, v in dino_test_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(CHECKPOINT_DIR / "test_results_dinov2.json", "w") as f:
    json.dump(dino_test_metrics, f, indent=2)

[DINOv2] Loaded best checkpoint from epoch 26 (val_f1_macro=0.9468)


  0%|          | 0/3 [00:00<?, ?it/s]


[DINOv2] Test set results:
  precision_macro: 0.9155
  recall_macro: 0.8333
  f1_macro: 0.8641
  mAP_macro: 0.9080
  subset_accuracy: 0.8333
  loss: 0.9017


In [18]:
# ================================================================
# 10b. MobileNetV3-Large — frozen ImageNet backbone + head
# ================================================================
# Self-contained: reuses train/val/test_paths, *_labels, compute_metrics,
# CCTVDataset, and the config constants from the CLIP cells above.

import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from tqdm.auto import tqdm

MOBILENET_FEAT_DIM = 960         # MobileNetV3-Large `features` output channels

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Model-specific preprocessing (ImageNet stats) -------------------------
# Our crops are already 224x224 (MobileNetV3's native size), so we only need
# ToTensor + the ImageNet normalization the pretrained weights expect.
mobilenet_normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
)
mobilenet_transform = transforms.Compose([transforms.ToTensor(), mobilenet_normalize])

mnet_train_ds = CCTVDataset(train_paths, train_labels, mobilenet_transform)
mnet_val_ds   = CCTVDataset(val_paths,   val_labels,   mobilenet_transform)
mnet_test_ds  = CCTVDataset(test_paths,  test_labels,  mobilenet_transform)

mnet_train_loader = DataLoader(mnet_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
mnet_val_loader   = DataLoader(mnet_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
mnet_test_loader  = DataLoader(mnet_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

# ---- Load pretrained backbone and FREEZE it --------------------------------
_mnet_pretrained = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
for p in _mnet_pretrained.parameters():
    p.requires_grad = False


class MobileNetV3Classifier(nn.Module):
    """Frozen MobileNetV3-Large feature extractor (`features` -> global pool ->
    960-d vector) + the same multi-label head used for the other models.
    The original ImageNet classifier is dropped."""
    def __init__(self, pretrained, feat_dim, num_labels):
        super().__init__()
        self.features = pretrained.features            # conv trunk (frozen)
        self.pool = nn.AdaptiveAvgPool2d(1)            # -> [B, feat_dim, 1, 1]
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def forward(self, x):
        with torch.no_grad():                          # backbone is frozen
            f = self.features(x)
            f = self.pool(f).flatten(1)                # [B, feat_dim]
        return self.head(f)


mnet_model = MobileNetV3Classifier(_mnet_pretrained, MOBILENET_FEAT_DIM, NUM_LABELS).to(DEVICE)

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    assert mnet_model(_dummy).shape == (1, NUM_LABELS)

_trainable = sum(p.numel() for p in mnet_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in mnet_model.parameters())
print(f"[MobileNetV3] Trainable params: {_trainable:,} / {_total:,} "
      f"({100 * _trainable / _total:.2f}%)")

# ---- Loss (pos_weight from THIS split's train balance), optimizer ----------
_train_arr = np.stack(train_labels)
_num_pos = _train_arr.sum(axis=0)
_num_neg = len(_train_arr) - _num_pos
mnet_pos_weight = torch.tensor(_num_neg / np.clip(_num_pos, 1, None),
                               dtype=torch.float32, device=DEVICE)
mnet_criterion = nn.BCEWithLogitsLoss(pos_weight=mnet_pos_weight)
mnet_optimizer = torch.optim.AdamW(
    (p for p in mnet_model.parameters() if p.requires_grad),
    lr=LR, weight_decay=WEIGHT_DECAY,
)


# ---- Train / validate with checkpointing (mirrors the CLIP loop) -----------
def mnet_run_epoch(loader, train: bool):
    mnet_model.train(train)
    total_loss = 0.0
    all_labels, all_logits = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = mnet_model(imgs)
            loss = mnet_criterion(logits, labels)
            if train:
                mnet_optimizer.zero_grad()
                loss.backward()
                mnet_optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_logits.append(logits.detach().cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_labels), np.concatenate(all_logits))
    metrics["loss"] = total_loss / len(loader.dataset)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics


mnet_history = []
mnet_best_f1 = -1.0
mnet_best_ckpt = CHECKPOINT_DIR / "best_model_mobilenetv3.pt"

EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    tr = mnet_run_epoch(mnet_train_loader, train=True)
    va = mnet_run_epoch(mnet_val_loader, train=False)
    print(f"[MobileNetV3][Epoch {epoch}/{EPOCHS}] "
          f"train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} "
          f"val_f1={va['f1_macro']:.4f} val_mAP={va['mAP_macro']:.4f}")
    mnet_history.append({"epoch": epoch, "train": tr, "val": va})
    with open(CHECKPOINT_DIR / "training_history_mobilenetv3.json", "w") as f:
        json.dump(mnet_history, f, indent=2)
    if va["f1_macro"] > mnet_best_f1:
        mnet_best_f1 = va["f1_macro"]
        torch.save({"epoch": epoch, "model_state_dict": mnet_model.state_dict(),
                    "val_f1_macro": mnet_best_f1}, mnet_best_ckpt)
        print(f"  ✓ New best (val_f1_macro={mnet_best_f1:.4f}) -> {mnet_best_ckpt}")

print(f"\n[MobileNetV3] Best val_f1_macro: {mnet_best_f1:.4f}")



Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 73.4MB/s]


[MobileNetV3] Trainable params: 248,072 / 3,220,024 (7.70%)


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

...

  0%|          | 0/3 [00:00<?, ?it/s]

[MobileNetV3][Epoch 29/30] train_loss=0.4146 val_loss=1.0401 val_f1=0.4393 val_mAP=0.5212


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[MobileNetV3][Epoch 30/30] train_loss=0.3973 val_loss=1.0456 val_f1=0.4222 val_mAP=0.5198

[MobileNetV3] Best val_f1_macro: 0.4716


In [19]:
# ---- Final test-set evaluation ---------------------------------------------
_ckpt = torch.load(mnet_best_ckpt, map_location=DEVICE)
mnet_model.load_state_dict(_ckpt["model_state_dict"])
print(f"[MobileNetV3] Loaded best checkpoint from epoch {_ckpt['epoch']} "
      f"(val_f1_macro={_ckpt['val_f1_macro']:.4f})")

mnet_test_metrics = mnet_run_epoch(mnet_test_loader, train=False)
print("\n[MobileNetV3] Test set results:")
for k, v in mnet_test_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(CHECKPOINT_DIR / "test_results_mobilenetv3.json", "w") as f:
    json.dump(mnet_test_metrics, f, indent=2)

[MobileNetV3] Loaded best checkpoint from epoch 19 (val_f1_macro=0.4716)


  0%|          | 0/3 [00:00<?, ?it/s]


[MobileNetV3] Test set results:
  precision_macro: 0.2888
  recall_macro: 0.4792
  f1_macro: 0.3535
  mAP_macro: 0.4143
  subset_accuracy: 0.1250
  loss: 1.2126


In [20]:
# ================================================================
# 10c. EfficientNetV2-S — frozen ImageNet backbone + head
# ================================================================
# Self-contained: reuses train/val/test_paths, *_labels, compute_metrics,
# CCTVDataset, and the config constants from the CLIP cells above.

import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from tqdm.auto import tqdm

EFFNET_FEAT_DIM = 1280           # EfficientNetV2-S `features` output channels

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Model-specific preprocessing (ImageNet stats) -------------------------
# EfficientNetV2-S's own eval preset resizes to 384, but the backbone is fully
# convolutional and we pool globally, so our fixed 224x224 crops feed through
# fine — and staying at 224 keeps the comparison to the other models on
# identical inputs. Only ToTensor + ImageNet normalization is needed.
effnet_normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
)
effnet_transform = transforms.Compose([transforms.ToTensor(), effnet_normalize])

eff_train_ds = CCTVDataset(train_paths, train_labels, effnet_transform)
eff_val_ds   = CCTVDataset(val_paths,   val_labels,   effnet_transform)
eff_test_ds  = CCTVDataset(test_paths,  test_labels,  effnet_transform)

eff_train_loader = DataLoader(eff_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
eff_val_loader   = DataLoader(eff_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
eff_test_loader  = DataLoader(eff_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

# ---- Load pretrained backbone and FREEZE it --------------------------------
_eff_pretrained = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
for p in _eff_pretrained.parameters():
    p.requires_grad = False


class EfficientNetV2Classifier(nn.Module):
    """Frozen EfficientNetV2-S feature extractor (`features` -> global pool ->
    1280-d vector) + the same multi-label head used for the other models.
    The original ImageNet classifier is dropped."""
    def __init__(self, pretrained, feat_dim, num_labels):
        super().__init__()
        self.features = pretrained.features            # conv trunk (frozen)
        self.pool = nn.AdaptiveAvgPool2d(1)            # -> [B, feat_dim, 1, 1]
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def forward(self, x):
        with torch.no_grad():                          # backbone is frozen
            f = self.features(x)
            f = self.pool(f).flatten(1)                # [B, feat_dim]
        return self.head(f)


eff_model = EfficientNetV2Classifier(_eff_pretrained, EFFNET_FEAT_DIM, NUM_LABELS).to(DEVICE)

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    assert eff_model(_dummy).shape == (1, NUM_LABELS)

_trainable = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in eff_model.parameters())
print(f"[EfficientNetV2] Trainable params: {_trainable:,} / {_total:,} "
      f"({100 * _trainable / _total:.2f}%)")

# ---- Loss (pos_weight from THIS split's train balance), optimizer ----------
_train_arr = np.stack(train_labels)
_num_pos = _train_arr.sum(axis=0)
_num_neg = len(_train_arr) - _num_pos
eff_pos_weight = torch.tensor(_num_neg / np.clip(_num_pos, 1, None),
                              dtype=torch.float32, device=DEVICE)
eff_criterion = nn.BCEWithLogitsLoss(pos_weight=eff_pos_weight)
eff_optimizer = torch.optim.AdamW(
    (p for p in eff_model.parameters() if p.requires_grad),
    lr=LR, weight_decay=WEIGHT_DECAY,
)


# ---- Train / validate with checkpointing (mirrors the CLIP loop) -----------
def eff_run_epoch(loader, train: bool):
    eff_model.train(train)
    total_loss = 0.0
    all_labels, all_logits = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = eff_model(imgs)
            loss = eff_criterion(logits, labels)
            if train:
                eff_optimizer.zero_grad()
                loss.backward()
                eff_optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_logits.append(logits.detach().cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_labels), np.concatenate(all_logits))
    metrics["loss"] = total_loss / len(loader.dataset)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics


eff_history = []
eff_best_f1 = -1.0
eff_best_ckpt = CHECKPOINT_DIR / "best_model_efficientnetv2.pt"

for epoch in range(1, EPOCHS + 1):
    tr = eff_run_epoch(eff_train_loader, train=True)
    va = eff_run_epoch(eff_val_loader, train=False)
    print(f"[EfficientNetV2][Epoch {epoch}/{EPOCHS}] "
          f"train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} "
          f"val_f1={va['f1_macro']:.4f} val_mAP={va['mAP_macro']:.4f}")
    eff_history.append({"epoch": epoch, "train": tr, "val": va})
    with open(CHECKPOINT_DIR / "training_history_efficientnetv2.json", "w") as f:
        json.dump(eff_history, f, indent=2)
    if va["f1_macro"] > eff_best_f1:
        eff_best_f1 = va["f1_macro"]
        torch.save({"epoch": epoch, "model_state_dict": eff_model.state_dict(),
                    "val_f1_macro": eff_best_f1}, eff_best_ckpt)
        print(f"  ✓ New best (val_f1_macro={eff_best_f1:.4f}) -> {eff_best_ckpt}")

print(f"\n[EfficientNetV2] Best val_f1_macro: {eff_best_f1:.4f}")


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 113MB/s]


[EfficientNetV2] Trainable params: 329,992 / 20,507,480 (1.61%)


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

...

  0%|          | 0/3 [00:00<?, ?it/s]

[EfficientNetV2][Epoch 29/30] train_loss=0.6271 val_loss=1.0751 val_f1=0.3494 val_mAP=0.3682


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[EfficientNetV2][Epoch 30/30] train_loss=0.6304 val_loss=1.1036 val_f1=0.3725 val_mAP=0.3733

[EfficientNetV2] Best val_f1_macro: 0.3988


In [21]:
# ---- Final test-set evaluation ---------------------------------------------
_ckpt = torch.load(eff_best_ckpt, map_location=DEVICE)
eff_model.load_state_dict(_ckpt["model_state_dict"])
print(f"[EfficientNetV2] Loaded best checkpoint from epoch {_ckpt['epoch']} "
      f"(val_f1_macro={_ckpt['val_f1_macro']:.4f})")

eff_test_metrics = eff_run_epoch(eff_test_loader, train=False)
print("\n[EfficientNetV2] Test set results:")
for k, v in eff_test_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(CHECKPOINT_DIR / "test_results_efficientnetv2.json", "w") as f:
    json.dump(eff_test_metrics, f, indent=2)

[EfficientNetV2] Loaded best checkpoint from epoch 12 (val_f1_macro=0.3988)


  0%|          | 0/3 [00:00<?, ?it/s]


[EfficientNetV2] Test set results:
  precision_macro: 0.2946
  recall_macro: 0.6667
  f1_macro: 0.3919
  mAP_macro: 0.5135
  subset_accuracy: 0.0833
  loss: 0.9910


In [23]:
# ================================================================
# 10d. SigLIP ViT-B/16 — frozen backbone + LoRA + head
# ================================================================
# Self-contained: reuses train/val/test_paths, *_labels, compute_metrics,
# CCTVDataset, and the config constants from the CLIP cells above.
#
# SigLIP is a CLIP-style vision-language model whose image tower is a ViT, so —
# exactly like DINOv2 — the faithful "repeat" is frozen backbone + LoRA + head.

!pip install -q transformers

import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

# ---- Free RAM / VRAM left over from previous model cells --------------------
# The earlier comparison cells leave large models + loaders resident; drop any
# that exist so SigLIP starts from a clean memory footprint (harmless if this is
# the first cell run — the names simply won't be defined).
for _name in ["clip_model", "model", "dino_model", "dino_backbone",
              "mnet_model", "_mnet_pretrained", "eff_model", "_eff_pretrained",
              "cnx_model", "convnext_backbone", "rnet_model"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SIGLIP_MODEL = "google/siglip-base-patch16-224"
SIGLIP_EMBED_DIM = 768           # SigLIP-base image-feature / projection dim

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Model-specific preprocessing (SigLIP uses 0.5/0.5/0.5, NOT ImageNet) --
# patch16-224 expects 224x224, which our crops already are.
siglip_normalize = transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
siglip_transform = transforms.Compose([transforms.ToTensor(), siglip_normalize])

sig_train_ds = CCTVDataset(train_paths, train_labels, siglip_transform)
sig_val_ds   = CCTVDataset(val_paths,   val_labels,   siglip_transform)
sig_test_ds  = CCTVDataset(test_paths,  test_labels,  siglip_transform)

sig_train_loader = DataLoader(sig_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
sig_val_loader   = DataLoader(sig_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
sig_test_loader  = DataLoader(sig_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

# ---- Load pretrained backbone, freeze it, inject LoRA ----------------------
sig_backbone = AutoModel.from_pretrained(SIGLIP_MODEL)
for p in sig_backbone.parameters():
    p.requires_grad = False

# SigLIP's transformer blocks use separate q/k/v/out projections plus an MLP
# with fc1/fc2 — discover the matching Linear leaf names from the model so we
# don't hardcode anything that could drift between transformers versions.
def _discover_siglip_lora_targets(
    module, suffixes=("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2")
):
    found = set()
    for name, sub in module.named_modules():
        if isinstance(sub, nn.Linear) and name.split(".")[-1] in suffixes:
            found.add(name.split(".")[-1])
    return sorted(found)

sig_targets = _discover_siglip_lora_targets(sig_backbone.vision_model)
print(f"SigLIP LoRA target modules: {sig_targets}")
assert sig_targets, "No Linear submodules matched — inspect sig_backbone.vision_model."

sig_lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=sig_targets,
    lora_dropout=LORA_DROPOUT, bias="none",
)
sig_backbone = get_peft_model(sig_backbone, sig_lora_cfg)


class SigLIPClassifier(nn.Module):
    """Frozen (LoRA-adapted) SigLIP image tower + multi-label head.

    We extract the image embedding from the UNDERLYING SiglipModel (via peft's
    get_base_model()), NOT through the PeftModel wrapper: the wrapper's __call__
    proxies to SiglipModel.forward, which returns a BaseModelOutputWithPooling
    object rather than the image-features tensor — that mismatch is what caused
    the `'BaseModelOutputWithPooling' has no attribute 'float'` error. The LoRA
    layers live inside the (base) vision tower, so calling the base model still
    routes through them. `get_image_features` returns a [B, 768] tensor; if a
    given transformers version ever returns an output object instead, we fall
    back to the vision tower's pooler_output (also [B, 768])."""
    def __init__(self, backbone, embed_dim, num_labels):
        super().__init__()
        self.base = backbone
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def _image_embeds(self, x):
        # peft: unwrap to the real SiglipModel (LoRA modules are swapped in-place
        # inside it, so this still exercises the adapters).
        core = self.base.get_base_model() if hasattr(self.base, "get_base_model") else self.base
        feats = core.get_image_features(pixel_values=x)
        if not torch.is_tensor(feats):                 # defensive: older/newer API
            feats = core.vision_model(pixel_values=x).pooler_output
        return feats

    def forward(self, x):
        return self.head(self._image_embeds(x).float())


sig_model = SigLIPClassifier(sig_backbone, SIGLIP_EMBED_DIM, NUM_LABELS).to(DEVICE)

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    _out = sig_model(_dummy)
    assert _out.shape == (1, NUM_LABELS), f"Unexpected output shape {_out.shape}"

_trainable = sum(p.numel() for p in sig_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in sig_model.parameters())
print(f"[SigLIP] Trainable params: {_trainable:,} / {_total:,} "
      f"({100 * _trainable / _total:.2f}%)")

# ---- Loss (pos_weight from THIS split's train balance), optimizer ----------
_train_arr = np.stack(train_labels)
_num_pos = _train_arr.sum(axis=0)
_num_neg = len(_train_arr) - _num_pos
sig_pos_weight = torch.tensor(_num_neg / np.clip(_num_pos, 1, None),
                              dtype=torch.float32, device=DEVICE)
sig_criterion = nn.BCEWithLogitsLoss(pos_weight=sig_pos_weight)
sig_optimizer = torch.optim.AdamW(
    (p for p in sig_model.parameters() if p.requires_grad),
    lr=LR, weight_decay=WEIGHT_DECAY,
)


# ---- Train / validate with checkpointing (mirrors the CLIP loop) -----------
def sig_run_epoch(loader, train: bool):
    sig_model.train(train)
    total_loss = 0.0
    all_labels, all_logits = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = sig_model(imgs)
            loss = sig_criterion(logits, labels)
            if train:
                sig_optimizer.zero_grad()
                loss.backward()
                sig_optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_logits.append(logits.detach().cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_labels), np.concatenate(all_logits))
    metrics["loss"] = total_loss / len(loader.dataset)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics


sig_history = []
sig_best_f1 = -1.0
sig_best_ckpt = CHECKPOINT_DIR / "best_model_siglip.pt"

for epoch in range(1, EPOCHS + 1):
    tr = sig_run_epoch(sig_train_loader, train=True)
    va = sig_run_epoch(sig_val_loader, train=False)
    print(f"[SigLIP][Epoch {epoch}/{EPOCHS}] "
          f"train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} "
          f"val_f1={va['f1_macro']:.4f} val_mAP={va['mAP_macro']:.4f}")
    sig_history.append({"epoch": epoch, "train": tr, "val": va})
    with open(CHECKPOINT_DIR / "training_history_siglip.json", "w") as f:
        json.dump(sig_history, f, indent=2)
    if va["f1_macro"] > sig_best_f1:
        sig_best_f1 = va["f1_macro"]
        torch.save({"epoch": epoch, "model_state_dict": sig_model.state_dict(),
                    "val_f1_macro": sig_best_f1}, sig_best_ckpt)
        print(f"  ✓ New best (val_f1_macro={sig_best_f1:.4f}) -> {sig_best_ckpt}")

print(f"\n[SigLIP] Best val_f1_macro: {sig_best_f1:.4f}")


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP LoRA target modules: ['fc1', 'fc2', 'k_proj', 'out_proj', 'q_proj', 'v_proj']
[SigLIP] Trainable params: 2,926,856 / 206,082,826 (1.42%)


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[SigLIP][Epoch 1/30] train_loss=1.0420 val_loss=0.6963 val_f1=0.5761 val_mAP=0.8009
  ✓ New best (val_f1_macro=0.5761) -> /content/checkpoints/best_model_siglip.pt


...

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[SigLIP][Epoch 17/30] train_loss=0.0003 val_loss=0.7530 val_f1=0.8853 val_mAP=0.9147


  0%|          | 0/84 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [24]:
# ---- Final test-set evaluation ---------------------------------------------
_ckpt = torch.load(sig_best_ckpt, map_location=DEVICE)
sig_model.load_state_dict(_ckpt["model_state_dict"])
print(f"[SigLIP] Loaded best checkpoint from epoch {_ckpt['epoch']} "
      f"(val_f1_macro={_ckpt['val_f1_macro']:.4f})")

sig_test_metrics = sig_run_epoch(sig_test_loader, train=False)
print("\n[SigLIP] Test set results:")
for k, v in sig_test_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(CHECKPOINT_DIR / "test_results_siglip.json", "w") as f:
    json.dump(sig_test_metrics, f, indent=2)

[SigLIP] Loaded best checkpoint from epoch 4 (val_f1_macro=0.8856)


  0%|          | 0/3 [00:00<?, ?it/s]


[SigLIP] Test set results:
  precision_macro: 0.8415
  recall_macro: 0.8958
  f1_macro: 0.8556
  mAP_macro: 0.8820
  subset_accuracy: 0.7500
  loss: 0.5457


In [25]:
# ================================================================
# 10e. ConvNeXt V2-Tiny — frozen ImageNet backbone + head
# ================================================================
# Self-contained: reuses train/val/test_paths, *_labels, compute_metrics,
# CCTVDataset, and the config constants from the CLIP cells above.
#
# ConvNeXt V2 is conv-based, so (per the CNN convention used for MobileNet /
# EfficientNet) it's frozen ImageNet backbone + head. torchvision only ships
# ConvNeXt V1, so we pull V2 from timm.

!pip install -q timm

import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import timm
from tqdm.auto import tqdm

CONVNEXT_MODEL = "convnextv2_tiny.fcmae_ft_in22k_in1k"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Load pretrained backbone (num_classes=0 -> pooled feature vector) ------
# num_classes=0 makes timm return the global-pooled embedding directly, so we
# don't have to pool by hand. Freeze all backbone params.
convnext_backbone = timm.create_model(CONVNEXT_MODEL, pretrained=True, num_classes=0)
for p in convnext_backbone.parameters():
    p.requires_grad = False

CONVNEXT_FEAT_DIM = convnext_backbone.num_features   # 768 for the tiny variant
print(f"ConvNeXtV2 feature dim: {CONVNEXT_FEAT_DIM}")

# ---- Model-specific preprocessing: use timm's OWN resolved mean/std ---------
# Reading them from the model's pretrained config guarantees we match exactly
# what these weights were trained with (ImageNet stats, but taken from source
# rather than hardcoded). Our crops are already 224x224.
_cfg = timm.data.resolve_model_data_config(convnext_backbone)
convnext_normalize = transforms.Normalize(mean=_cfg["mean"], std=_cfg["std"])
print(f"ConvNeXtV2 normalize: mean={_cfg['mean']}, std={_cfg['std']}")
convnext_transform = transforms.Compose([transforms.ToTensor(), convnext_normalize])

cnx_train_ds = CCTVDataset(train_paths, train_labels, convnext_transform)
cnx_val_ds   = CCTVDataset(val_paths,   val_labels,   convnext_transform)
cnx_test_ds  = CCTVDataset(test_paths,  test_labels,  convnext_transform)

cnx_train_loader = DataLoader(cnx_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
cnx_val_loader   = DataLoader(cnx_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
cnx_test_loader  = DataLoader(cnx_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))


class ConvNeXtV2Classifier(nn.Module):
    """Frozen ConvNeXt V2 feature extractor (num_classes=0 -> pooled vector) +
    the same multi-label head used for the other models."""
    def __init__(self, backbone, feat_dim, num_labels):
        super().__init__()
        self.base = backbone
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def forward(self, x):
        with torch.no_grad():                          # backbone is frozen
            f = self.base(x)                           # [B, feat_dim]
        return self.head(f)


cnx_model = ConvNeXtV2Classifier(convnext_backbone, CONVNEXT_FEAT_DIM, NUM_LABELS).to(DEVICE)

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    assert cnx_model(_dummy).shape == (1, NUM_LABELS)

_trainable = sum(p.numel() for p in cnx_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in cnx_model.parameters())
print(f"[ConvNeXtV2] Trainable params: {_trainable:,} / {_total:,} "
      f"({100 * _trainable / _total:.2f}%)")

# ---- Loss (pos_weight from THIS split's train balance), optimizer ----------
_train_arr = np.stack(train_labels)
_num_pos = _train_arr.sum(axis=0)
_num_neg = len(_train_arr) - _num_pos
cnx_pos_weight = torch.tensor(_num_neg / np.clip(_num_pos, 1, None),
                              dtype=torch.float32, device=DEVICE)
cnx_criterion = nn.BCEWithLogitsLoss(pos_weight=cnx_pos_weight)
cnx_optimizer = torch.optim.AdamW(
    (p for p in cnx_model.parameters() if p.requires_grad),
    lr=LR, weight_decay=WEIGHT_DECAY,
)


# ---- Train / validate with checkpointing (mirrors the CLIP loop) -----------
def cnx_run_epoch(loader, train: bool):
    cnx_model.train(train)
    total_loss = 0.0
    all_labels, all_logits = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = cnx_model(imgs)
            loss = cnx_criterion(logits, labels)
            if train:
                cnx_optimizer.zero_grad()
                loss.backward()
                cnx_optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_logits.append(logits.detach().cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_labels), np.concatenate(all_logits))
    metrics["loss"] = total_loss / len(loader.dataset)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics


cnx_history = []
cnx_best_f1 = -1.0
cnx_best_ckpt = CHECKPOINT_DIR / "best_model_convnextv2.pt"

for epoch in range(1, EPOCHS + 1):
    tr = cnx_run_epoch(cnx_train_loader, train=True)
    va = cnx_run_epoch(cnx_val_loader, train=False)
    print(f"[ConvNeXtV2][Epoch {epoch}/{EPOCHS}] "
          f"train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} "
          f"val_f1={va['f1_macro']:.4f} val_mAP={va['mAP_macro']:.4f}")
    cnx_history.append({"epoch": epoch, "train": tr, "val": va})
    with open(CHECKPOINT_DIR / "training_history_convnextv2.json", "w") as f:
        json.dump(cnx_history, f, indent=2)
    if va["f1_macro"] > cnx_best_f1:
        cnx_best_f1 = va["f1_macro"]
        torch.save({"epoch": epoch, "model_state_dict": cnx_model.state_dict(),
                    "val_f1_macro": cnx_best_f1}, cnx_best_ckpt)
        print(f"  ✓ New best (val_f1_macro={cnx_best_f1:.4f}) -> {cnx_best_ckpt}")

print(f"\n[ConvNeXtV2] Best val_f1_macro: {cnx_best_f1:.4f}")


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

ConvNeXtV2 feature dim: 768
ConvNeXtV2 normalize: mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
[ConvNeXtV2] Trainable params: 198,920 / 28,065,416 (0.71%)


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[ConvNeXtV2][Epoch 1/30] train_loss=1.0946 val_loss=1.0814 val_f1=0.3455 val_mAP=0.4186
  ✓ New best (val_f1_macro=0.3455) -> /content/checkpoints/best_model_convnextv2.pt


...

  0%|          | 0/3 [00:00<?, ?it/s]

[ConvNeXtV2][Epoch 29/30] train_loss=0.1140 val_loss=0.9750 val_f1=0.5837 val_mAP=0.6515


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[ConvNeXtV2][Epoch 30/30] train_loss=0.1074 val_loss=0.9638 val_f1=0.5967 val_mAP=0.6562

[ConvNeXtV2] Best val_f1_macro: 0.6011


In [26]:
# ---- Final test-set evaluation ---------------------------------------------
_ckpt = torch.load(cnx_best_ckpt, map_location=DEVICE)
cnx_model.load_state_dict(_ckpt["model_state_dict"])
print(f"[ConvNeXtV2] Loaded best checkpoint from epoch {_ckpt['epoch']} "
      f"(val_f1_macro={_ckpt['val_f1_macro']:.4f})")

cnx_test_metrics = cnx_run_epoch(cnx_test_loader, train=False)
print("\n[ConvNeXtV2] Test set results:")
for k, v in cnx_test_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(CHECKPOINT_DIR / "test_results_convnextv2.json", "w") as f:
    json.dump(cnx_test_metrics, f, indent=2)

[ConvNeXtV2] Loaded best checkpoint from epoch 28 (val_f1_macro=0.6011)


  0%|          | 0/3 [00:00<?, ?it/s]


[ConvNeXtV2] Test set results:
  precision_macro: 0.6547
  recall_macro: 0.6667
  f1_macro: 0.6350
  mAP_macro: 0.7061
  subset_accuracy: 0.5000
  loss: 1.0036


In [27]:
# ================================================================
# 10f. ResNet-18 — trained FROM SCRATCH (no pretrained weights)
# ================================================================
# Self-contained: reuses train/val/test_paths, *_labels, compute_metrics,
# CCTVDataset, and the config constants from the CLIP cells above.
#
# Unlike every other model here, this one is a baseline with NO pretrained
# weights: the whole network is randomly initialized and FULLY trainable (no
# frozen backbone, no LoRA). Same data / head / loss / optimizer / EPOCHS / LR
# as the others, so it's an apples-to-apples "how far does the pretraining get
# you" reference point. Expect it to underperform the pretrained backbones on
# this small dataset — that gap is the finding.

import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import resnet18
from tqdm.auto import tqdm

RESNET_FEAT_DIM = 512            # ResNet-18 penultimate (pre-fc) feature dim

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Preprocessing --------------------------------------------------------
# No pretrained weights means no "expected" normalization; ImageNet mean/std is
# a conventional, harmless input-centering choice (the net learns its own scale
# regardless). Kept identical in spirit to the other cells for comparability.
# Crops are already 224x224.
resnet_normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
)
resnet_transform = transforms.Compose([transforms.ToTensor(), resnet_normalize])

rnet_train_ds = CCTVDataset(train_paths, train_labels, resnet_transform)
rnet_val_ds   = CCTVDataset(val_paths,   val_labels,   resnet_transform)
rnet_test_ds  = CCTVDataset(test_paths,  test_labels,  resnet_transform)

rnet_train_loader = DataLoader(rnet_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
rnet_val_loader   = DataLoader(rnet_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
rnet_test_loader  = DataLoader(rnet_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))


class ResNet18Scratch(nn.Module):
    """ResNet-18 from random init, fully trainable, with its final fc replaced
    by the same multi-label head used for the other models. The backbone is NOT
    frozen — gradients flow through everything."""
    def __init__(self, feat_dim, num_labels):
        super().__init__()
        # weights=None -> random initialization (train from scratch).
        self.backbone = resnet18(weights=None)
        # Drop the 1000-way ImageNet fc; forward then yields the 512-d features.
        self.backbone.fc = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def forward(self, x):
        f = self.backbone(x)                 # [B, 512] — trainable, grads flow
        return self.head(f)


rnet_model = ResNet18Scratch(RESNET_FEAT_DIM, NUM_LABELS).to(DEVICE)

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    assert rnet_model(_dummy).shape == (1, NUM_LABELS)

_trainable = sum(p.numel() for p in rnet_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in rnet_model.parameters())
print(f"[ResNet18-scratch] Trainable params: {_trainable:,} / {_total:,} "
      f"({100 * _trainable / _total:.2f}%)  <- ~100%, nothing frozen")

# ---- Loss (pos_weight from THIS split's train balance), optimizer ----------
_train_arr = np.stack(train_labels)
_num_pos = _train_arr.sum(axis=0)
_num_neg = len(_train_arr) - _num_pos
rnet_pos_weight = torch.tensor(_num_neg / np.clip(_num_pos, 1, None),
                               dtype=torch.float32, device=DEVICE)
rnet_criterion = nn.BCEWithLogitsLoss(pos_weight=rnet_pos_weight)
rnet_optimizer = torch.optim.AdamW(
    (p for p in rnet_model.parameters() if p.requires_grad),   # = all params
    lr=LR, weight_decay=WEIGHT_DECAY,
)


# ---- Train / validate with checkpointing (mirrors the CLIP loop) -----------
def rnet_run_epoch(loader, train: bool):
    rnet_model.train(train)
    total_loss = 0.0
    all_labels, all_logits = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = rnet_model(imgs)
            loss = rnet_criterion(logits, labels)
            if train:
                rnet_optimizer.zero_grad()
                loss.backward()
                rnet_optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_logits.append(logits.detach().cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_labels), np.concatenate(all_logits))
    metrics["loss"] = total_loss / len(loader.dataset)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics


rnet_history = []
rnet_best_f1 = -1.0
rnet_best_ckpt = CHECKPOINT_DIR / "best_model_resnet18_scratch.pt"

for epoch in range(1, EPOCHS + 1):
    tr = rnet_run_epoch(rnet_train_loader, train=True)
    va = rnet_run_epoch(rnet_val_loader, train=False)
    print(f"[ResNet18-scratch][Epoch {epoch}/{EPOCHS}] "
          f"train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} "
          f"val_f1={va['f1_macro']:.4f} val_mAP={va['mAP_macro']:.4f}")
    rnet_history.append({"epoch": epoch, "train": tr, "val": va})
    with open(CHECKPOINT_DIR / "training_history_resnet18_scratch.json", "w") as f:
        json.dump(rnet_history, f, indent=2)
    if va["f1_macro"] > rnet_best_f1:
        rnet_best_f1 = va["f1_macro"]
        torch.save({"epoch": epoch, "model_state_dict": rnet_model.state_dict(),
                    "val_f1_macro": rnet_best_f1}, rnet_best_ckpt)
        print(f"  ✓ New best (val_f1_macro={rnet_best_f1:.4f}) -> {rnet_best_ckpt}")

print(f"\n[ResNet18-scratch] Best val_f1_macro: {rnet_best_f1:.4f}")



[ResNet18-scratch] Trainable params: 11,309,896 / 11,309,896 (100.00%)  <- ~100%, nothing frozen


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[ResNet18-scratch][Epoch 1/30] train_loss=1.1135 val_loss=1.0097 val_f1=0.3673 val_mAP=0.4491
  ✓ New best (val_f1_macro=0.3673) -> /content/checkpoints/best_model_resnet18_scratch.pt


  0%|          | 0/84 [00:00<?, ?it/s]

...

  0%|          | 0/3 [00:00<?, ?it/s]

[ResNet18-scratch][Epoch 29/30] train_loss=0.0097 val_loss=2.0227 val_f1=0.5280 val_mAP=0.5678


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

[ResNet18-scratch][Epoch 30/30] train_loss=0.0417 val_loss=4.2075 val_f1=0.2042 val_mAP=0.4165

[ResNet18-scratch] Best val_f1_macro: 0.5308


In [28]:
# ---- Final test-set evaluation ---------------------------------------------
_ckpt = torch.load(rnet_best_ckpt, map_location=DEVICE)
rnet_model.load_state_dict(_ckpt["model_state_dict"])
print(f"[ResNet18-scratch] Loaded best checkpoint from epoch {_ckpt['epoch']} "
      f"(val_f1_macro={_ckpt['val_f1_macro']:.4f})")

rnet_test_metrics = rnet_run_epoch(rnet_test_loader, train=False)
print("\n[ResNet18-scratch] Test set results:")
for k, v in rnet_test_metrics.items():
    print(f"  {k}: {v:.4f}")
with open(CHECKPOINT_DIR / "test_results_resnet18_scratch.json", "w") as f:
    json.dump(rnet_test_metrics, f, indent=2)

[ResNet18-scratch] Loaded best checkpoint from epoch 27 (val_f1_macro=0.5308)


  0%|          | 0/3 [00:00<?, ?it/s]


[ResNet18-scratch] Test set results:
  precision_macro: 0.4754
  recall_macro: 0.4583
  f1_macro: 0.4436
  mAP_macro: 0.4360
  subset_accuracy: 0.3750
  loss: 2.7939
